# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a guided approach for loading and exploring the FAIR² Mental Health Survey Dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}\n{metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs using Croissant schema conventions. All entities are referenced by their `@id` fields.

### Enumerate Record Sets, Fields, and Columns
We will list the available record sets (`cr:recordSet`), fields (`cr:field`), and columns (`cr:column`) using their `@id`s. This is critical for referencing data correctly.

In [ ]:
# Inspect record sets and their fields/columns
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record sets.")

for rs in record_sets:
    rs_id = rs['@id'] if '@id' in rs else '[ID missing]'
    rs_name = rs.get('name', '[name missing]')
    print(f"RecordSet @id: {rs_id} | Name: {rs_name}")
    fields = rs.get('field', [])
    if not fields:
        print("  No fields defined.")
    else:
        for f in fields:
            fid = f['@id'] if '@id' in f else '[ID missing]'
            fname = f.get('name', '[name missing]')
            print(f"  Field @id: {fid} | Name: {fname}")
            cols = f.get('column', [])
            if not cols:
                print("    No columns.")
            else:
                for c in cols:
                    cid = c['@id'] if '@id' in c else '[ID missing]'
                    cname = c.get('name', '[name missing]')
                    print(f"    Column @id: {cid} | Name: {cname}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the record set and field `@id`s from the overview. Below, we demonstrate extracting all record sets found.

In [ ]:
# Extract data from each record set
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"RecordSet @id: {rs_id} | DataFrame shape: {df.shape}")

# Print columns of the first record set (if any)
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"Columns in RecordSet @id '{first_rs_id}':")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, we demonstrate filtering and normalizing a numeric field, and grouping by a categorical field. All field references are via `@id` as per Croissant convention.

In [ ]:
# Example: Filter and normalize a numeric field, group by categorical field
# Choose the first record set for this demo
if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    print(f"DataFrame loaded for RecordSet @id: {record_set_id}")
    
    # Find a numeric column by inspecting columns (e.g., GAD-7_score or PHQ-9_score, referenced by @id)
    numeric_candidate_ids = [col for col in df.columns if 'score' in col.lower() or 'GAD-7' in col or 'PHQ-9' in col or 'PSQ' in col]
    if numeric_candidate_ids:
        numeric_field_id = numeric_candidate_ids[0]
        print(f"Using numeric field @id: {numeric_field_id}")
        
        # Filter records above a threshold
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize the numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by a categorical field (e.g., gender or level_of_education by @id)
        group_candidates = [col for col in df.columns if ('gender' in col.lower() or 'education' in col.lower())]
        if group_candidates:
            group_field_id = group_candidates[0]
            print(f"Grouping by field @id: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No groupable categorical field found.")
    else:
        print("No numeric field candidates found in DataFrame.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib and seaborn.

Below, we plot the distribution of a numeric field and show how it varies by a group (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization of numeric variable distribution and grouping
if record_set_ids:
    df = dataframes[record_set_ids[0]]

    # Use the same numeric_field_id and group_field_id from EDA (if available)
    numeric_field = None
    group_field = None
    numeric_candidate_ids = [col for col in df.columns if 'score' in col.lower() or 'GAD-7' in col or 'PHQ-9' in col or 'PSQ' in col]
    group_candidates = [col for col in df.columns if ('gender' in col.lower() or 'education' in col.lower())]

    if numeric_candidate_ids:
        numeric_field = numeric_candidate_ids[0]
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field], bins=15, kde=True)
        plt.title(f'Distribution of {numeric_field}')
        plt.xlabel(numeric_field)
        plt.ylabel('Frequency')
        plt.show()

        if group_candidates:
            group_field = group_candidates[0]
            plt.figure(figsize=(10, 5))
            sns.boxplot(x=group_field, y=numeric_field, data=df)
            plt.title(f'{numeric_field} by {group_field}')
            plt.show()
    else:
        print('No numeric field found for visualization.')
else:
    print('No record sets available for visualization.')

## 6. Conclusion
We have successfully loaded the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using `mlcroissant`, explored its schema using `@id`-based references, extracted record sets, performed basic EDA steps (filtering, normalization, grouping), and visualized distributions.

Further analysis can include exploring correlations among scores or linking demographic details to psychological indicators. The use of Croissant `@id` references ensures reproducibility and clarity when interacting with the dataset.